In [ ]:
from trajectory_aware_gym.config import AWSConfig, GEPAConfig, OllamaConfig

ollama = OllamaConfig()
aws = AWSConfig()
gepa = GEPAConfig()

print(f"Task model 1.7B: {ollama.local_task_model_1_7b}")
print(f"Task model 4B: {ollama.local_task_model_4b}")
print(f"Ollama base: {ollama.ollama_api_base}")
print(f"Reflection model: {gepa.gepa_reflection_model}")

Task model 1.7B: ollama_chat/qwen3:1.7b
Task model 4B: ollama_chat/qwen3:4b
Ollama base: http://localhost:11434
Reflection model: anthropic.claude-sonnet-4-5-v2:0


In [2]:
from litellm import completion

ollama = OllamaConfig()
response = completion(
    model=ollama.local_task_model_1_7b,
    messages=[{"role": "user", "content": "What is 2+2?"}],
    api_base=ollama.ollama_api_base,
    max_tokens=500,
    temperature=0,
)
print(f"Response: {response.choices[0].message.content}")
print(repr(response.choices[0].message.content))
print(f"Tokens: {response.usage.total_tokens}")

Response: The result of 2 + 2 is **4**. 

This is a basic arithmetic operation where two numbers are added together. In this case, 2 + 2 equals 4. 

Let me know if you'd like help with more math problems! 😊
"The result of 2 + 2 is **4**. \n\nThis is a basic arithmetic operation where two numbers are added together. In this case, 2 + 2 equals 4. \n\nLet me know if you'd like help with more math problems! 😊"
Tokens: 223


In [ ]:
import dspy

ollama = OllamaConfig()
task_lm = dspy.LM(
    model=ollama.local_task_model_1_7b,
    api_base=ollama.ollama_api_base,
    temperature=0,
    max_tokens=500,
)
dspy.configure(lm=task_lm)


class BasicQA(dspy.Signature):
    """Answer questions concisely."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


qa = dspy.ChainOfThought(BasicQA)
result = qa(question="What is 2+2?")
print(f"Answer: {result.answer}")

Answer: 4


In [ ]:
import dspy

# Test LLM Provider Factory
from trajectory_aware_gym.config.llm_provider import get_task_lm

# Get task model via factory (Qwen3 1.7B, eval mode = temp 0.0)
task_lm = get_task_lm("qwen3:1.7b", "eval")
print(f"Task model: {task_lm.model}")
print(f"Temperature: {task_lm.kwargs['temperature']}")
print(f"Max tokens: {task_lm.kwargs['max_tokens']}")

# Verify it works with DSPy
dspy.configure(lm=task_lm)


class FactCheck(dspy.Signature):
    """Answer with a single word or number."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


qa = dspy.Predict(FactCheck)
result = qa(question="What is the capital of France?")
print(f"\nFactory test - Answer: {result.answer}")

Task model: ollama_chat/qwen3:1.7b
Temperature: 0.0
Max tokens: 4096

Factory test - Answer: Paris[[ ## completed ## ]]
